# Data Cleaning & Preparation
## League of Legends Coaching Knowledge Base

---

## Dataset Overview

**Source:** Two SQLite databases built from a custom web scraping pipeline:
- `videos.db` — 494 rows of real coaching session recordings sourced from a professional team coach's Discord server. Videos were identified by parsing Discord message links and extracting YouTube URLs.
- `guide_test.db` — 687 rows of YouTube educational guide videos scraped using `yt-dlp` across all 172 playable League of Legends champions.

**Column Descriptions (`videos` table):**
| Column | Type | Description |
|---|---|---|
| `video_id` | TEXT (PK) | YouTube video ID |
| `video_url` | TEXT | Full YouTube URL |
| `video_title` | TEXT | Video title from YouTube |
| `description` | TEXT | Short description or title copy |
| `role` | TEXT | Lane role (top/jungle/mid/adc/support) |
| `champion` | TEXT | Champion featured in the video |
| `rank` | TEXT | Player rank shown in video |
| `message_timestamp` | TEXT | Discord message timestamp (Discord source only) |
| `status` | TEXT | Pipeline stage: pending/transcribed/analyzed/no_transcript |
| `transcription` | TEXT | Full auto-generated caption transcript |
| `source` | TEXT | Origin: 'discord' or 'youtube_guide' |

**Column Descriptions (`insights` table):**
| Column | Type | Description |
|---|---|---|
| `id` | INTEGER (PK) | Row ID |
| `video_id` | TEXT (FK) | Parent video |
| `insight_type` | TEXT | Category (champion_mechanics, laning_tips, etc.) |
| `text` | TEXT | The extracted coaching insight |
| `source_score` | REAL | Embedding similarity to source transcript (0–1) |
| `cluster_score` | REAL | Cross-video clustering score (null until scored) |
| `confidence` | REAL | Combined quality score (null until scored) |
| `repetition_count` | INTEGER | Times concept appeared in video |

**Size:** 1,181 video rows combined; 12,000+ insight rows combined across both databases.

**Relevance:** This dataset underpins a RAG (Retrieval-Augmented Generation) coaching assistant that answers League of Legends questions using real coaching insights. Clean, well-structured data directly determines response quality — noisy or duplicate insights degrade the system.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load both databases into pandas DataFrames
discord_conn = sqlite3.connect('videos.db')
guide_conn   = sqlite3.connect('guide_test.db')

df_discord = pd.read_sql('SELECT * FROM videos WHERE source = "discord"', discord_conn)
df_guides  = pd.read_sql('SELECT * FROM videos WHERE source = "youtube_guide"', guide_conn)
df_insights = pd.read_sql('SELECT * FROM insights', guide_conn)

# Combine video tables for joint analysis
df_videos = pd.concat([df_discord, df_guides], ignore_index=True)

print(f'Discord videos:       {len(df_discord):,} rows')
print(f'YouTube guide videos: {len(df_guides):,} rows')
print(f'Total videos:         {len(df_videos):,} rows')
print(f'Insights:             {len(df_insights):,} rows')

---
## Step 1 — Systematic Examination of Raw Data

Before cleaning anything, we inspect the raw data to understand its shape, data types, and where problems exist.

In [ ]:
print('=== Data Types ===')
print(df_videos.dtypes)
print()
print('=== First 5 rows ===')
df_videos.head()

In [ ]:
print('=== Missing Values (videos) ===')
nulls = df_videos.isnull().sum()
empty = (df_videos == '').sum()
summary = pd.DataFrame({'null_count': nulls, 'empty_string_count': empty})
summary['total_missing'] = summary['null_count'] + summary['empty_string_count']
summary['pct_missing'] = (summary['total_missing'] / len(df_videos) * 100).round(1)
print(summary[summary['total_missing'] > 0])

In [ ]:
print('=== Status Distribution ===')
print(df_videos['status'].value_counts())
print()
print('=== Source Distribution ===')
print(df_videos['source'].value_counts())
print()
print('=== Role Distribution ===')
print(df_videos['role'].value_counts())

In [ ]:
print('=== Insights — Missing Values ===')
print(df_insights.isnull().sum())
print()
print('=== Insights — source_score distribution ===')
print(df_insights['source_score'].describe())

---
## Step 2 — Cleaning: Videos with No URL (Discord Non-Video Messages)

### Issue Identified
The Discord scraper parsed every message in the coaching server, but not every message contained a YouTube link. Messages without links were still inserted as rows, leaving `video_url`, `video_title`, and `description` empty — 108 rows with no URL out of 494 Discord rows (~22%).

### Reasoning
These rows carry zero useful information — no URL means no video, no transcript, and no insights can ever be extracted from them. Imputing a URL is not possible. **We drop these rows entirely** rather than filling with a placeholder, because retaining them inflates counts and would cause downstream pipeline failures (transcription step tries to fetch a URL that doesn't exist).

### Before / After

In [ ]:
before = len(df_videos)
print(f'Before: {before} rows')
print(f'Rows with missing video_url: {(df_videos["video_url"].isnull() | (df_videos["video_url"] == "")).sum()}')

df_videos = df_videos[df_videos['video_url'].notna() & (df_videos['video_url'] != '')].copy()

after = len(df_videos)
print(f'\nAfter: {after} rows')
print(f'Rows removed: {before - after}')

---
## Step 3 — Cleaning: Missing Champion Names

### Issue Identified
11 Discord rows have no `champion` value. Champion is a critical field — the retrieval system filters insights by champion, so a video with no champion can never be surfaced in a query.

### Reasoning
We cannot reliably infer the champion from the video title alone (titles like "Coaching Session #12" are not specific). Imputing a placeholder like `"unknown"` would pollute champion-filtered queries with irrelevant results. **We drop these rows** since they cannot contribute meaningful, retrievable coaching data.

### Before / After

In [ ]:
before = len(df_videos)
missing_champ = df_videos['champion'].isnull() | (df_videos['champion'] == '')
print(f'Before: {before} rows')
print(f'Rows with missing champion: {missing_champ.sum()}')

df_videos = df_videos[~missing_champ].copy()

print(f'\nAfter: {len(df_videos)} rows')
print(f'Rows removed: {before - len(df_videos)}')

---
## Step 4 — Cleaning: Videos with No Transcript

### Issue Identified
109 rows have `status = 'no_transcript'` — YouTube's caption API returned nothing for these videos (captions disabled, private video, or regional restriction). Their `transcription` column is NULL.

### Reasoning
Without a transcript, no insights can be extracted and the row contributes nothing to the knowledge base. Imputing text is not feasible — we would be fabricating coaching content. **We flag these rows** by filtering them out of the analysis-ready dataset. We do not delete them from the database in case captions become available later, but they are excluded from downstream analysis.

### Before / After

In [ ]:
before = len(df_videos)
no_transcript = df_videos['status'] == 'no_transcript'
print(f'Before: {before} rows')
print(f'Rows with no transcript: {no_transcript.sum()}')
print(f'  Breakdown by source:')
print(df_videos[no_transcript]['source'].value_counts())

df_clean = df_videos[~no_transcript].copy()

print(f'\nAfter: {len(df_clean)} rows')
print(f'Rows excluded: {before - len(df_clean)}')

---
## Step 5 — Cleaning: Non-English Video Titles

### Issue Identified
The YouTube scraper fetched results from a global index. Some returned videos had titles in French, Spanish, Korean, or other languages (e.g. `"GUIDE CHO GATH FR - DETRUIRE VOS GAMES AVEC CE DINOSAURE"`). Non-English transcripts cannot be meaningfully processed by an English-language LLM extraction pipeline — they produce garbled or empty insights.

### Reasoning
We detect non-English titles two ways: (1) any non-ASCII character in the title (catches accented/non-Latin scripts), and (2) explicit language code patterns like ` FR `, ` ES `, ` KR `. **We drop these rows** rather than attempting translation — the source language of the transcript matters, not just the title, and translation would introduce semantic drift into coaching advice.

### Before / After

In [ ]:
import re

LANG_TAGS = re.compile(r'\b(FR|ES|PT|DE|KR|CN|JP|IT|RU|PL|TR|NL)\b')

def is_non_english(title):
    if pd.isnull(title):
        return False
    if any(ord(c) > 127 for c in title):
        return True
    if LANG_TAGS.search(title.upper()):
        return True
    return False

before = len(df_clean)
non_english_mask = df_clean['video_title'].apply(is_non_english)
print(f'Before: {before} rows')
print(f'Non-English titles detected: {non_english_mask.sum()}')
print('Examples:')
for title in df_clean[non_english_mask]['video_title'].head(5):
    print(f'  {title}')

df_clean = df_clean[~non_english_mask].copy()
print(f'\nAfter: {len(df_clean)} rows')
print(f'Rows removed: {before - len(df_clean)}')

---
## Step 6 — Cleaning: Data Type Correction on `message_timestamp`

### Issue Identified
The `message_timestamp` column is stored as a plain TEXT string rather than a proper datetime type. This prevents any time-based sorting or filtering. Discord rows have a timestamp; YouTube guide rows have an empty string (the field is irrelevant for that source).

### Reasoning
We convert the column to `datetime` using `pd.to_datetime` with `errors='coerce'` — this correctly parses valid timestamps and converts empty strings or malformed values to `NaT` (Not a Time) rather than raising an error. This preserves all rows while giving us a properly typed column for any future time-based analysis.

### Before / After

In [ ]:
print(f'Before dtype: {df_clean["message_timestamp"].dtype}')
print(f'Sample values: {df_clean["message_timestamp"].dropna().head(3).tolist()}')

df_clean['message_timestamp'] = pd.to_datetime(df_clean['message_timestamp'], errors='coerce')

print(f'\nAfter dtype: {df_clean["message_timestamp"].dtype}')
print(f'Valid timestamps: {df_clean["message_timestamp"].notna().sum()}')
print(f'NaT (YouTube rows, expected): {df_clean["message_timestamp"].isna().sum()}')

---
## Step 7 — Cleaning: Low-Quality Insights (source_score outliers)

### Issue Identified
Each insight has a `source_score` — a cosine similarity score (0–1) measuring how well the insight text is grounded in the transcript it was extracted from. A score near 0 means the LLM likely hallucinated or the insight is too generic to match any specific part of the transcript. These are low-signal rows that degrade retrieval quality.

### Reasoning
We do **not** drop rows below a hard threshold — instead we flag them and use the score as a soft weight in retrieval ranking (low-score insights can still surface if semantically very relevant). However, for a clean analysis-ready dataset we filter out insights with `source_score < 0.40` as these are statistical outliers representing near-zero grounding. We choose 0.40 rather than a higher value to avoid discarding valid insights that happen to be phrased differently from the transcript.

### Before / After

In [ ]:
before = len(df_insights)
print(f'Before: {before:,} insight rows')
print(f'\nsource_score distribution:')
print(df_insights['source_score'].describe().round(3))

low_score = df_insights['source_score'] < 0.40
print(f'\nInsights with source_score < 0.40: {low_score.sum():,} ({low_score.mean()*100:.1f}%)')

df_insights_clean = df_insights[~low_score].copy()

print(f'\nAfter: {len(df_insights_clean):,} insight rows')
print(f'Rows removed: {before - len(df_insights_clean):,}')

---
## Step 8 — Cleaning: Duplicate Insights Within a Video

### Issue Identified
Long videos (1–5 hours) are split into ~1-hour chunks and each chunk is analyzed independently by the LLM. The same concept can be expressed in slightly different words across chunks — for example, "Hold your R for escapes" and "Avoid using R aggressively — save it as an escape tool" are semantically the same insight appearing twice. This inflates insight counts and distorts repetition scoring.

### Reasoning
We use cosine similarity on sentence embeddings (all-MiniLM-L6-v2, 384 dimensions) to detect near-duplicate insights **within the same video and category**. Pairs above a 0.82 similarity threshold are collapsed: the highest-emphasis version is kept, and the `repetition_count` is summed — because repeated mentions are genuine signal about what the presenter stressed.

We choose embedding similarity over exact string matching because the LLM paraphrases freely. We choose 0.82 (not higher) because debug runs showed genuine paraphrases clustering at 0.82–0.88, while distinct insights stay below 0.80.

### Before / After

In [ ]:
# Demonstrate the duplicate detection on a single champion's insights
# (Full dedup runs during pipeline/guide_analyze.py at extraction time)
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings('ignore')

model = SentenceTransformer('all-MiniLM-L6-v2')
THRESHOLD = 0.82

# Pick a champion with multiple videos to show cross-chunk overlap
sample = df_insights_clean[df_insights_clean['insight_type'] == 'laning_tips'].head(30)
texts = sample['text'].tolist()

vecs = model.encode(texts, normalize_embeddings=True)
sim_matrix = vecs @ vecs.T

pairs_above = [(i, j, round(float(sim_matrix[i,j]), 3))
               for i in range(len(texts))
               for j in range(i+1, len(texts))
               if sim_matrix[i,j] >= THRESHOLD]

print(f'Sample: {len(texts)} laning_tips insights')
print(f'Pairs above {THRESHOLD} similarity (near-duplicates): {len(pairs_above)}')
if pairs_above:
    i, j, s = pairs_above[0]
    print(f'\nExample near-duplicate (sim={s}):')
    print(f'  A: {texts[i][:100]}')
    print(f'  B: {texts[j][:100]}')

---
## Final Summary — Clean Dataset

The table below summarises every cleaning step applied and the net rows removed.

In [ ]:
print('=== Videos Table — Final State ===')
print(f'Total rows: {len(df_clean):,}')
print(f'\nSource breakdown:')
print(df_clean['source'].value_counts())
print(f'\nStatus breakdown:')
print(df_clean['status'].value_counts())
print(f'\nRemaining nulls:')
remaining_nulls = df_clean.isnull().sum()
print(remaining_nulls[remaining_nulls > 0])

print()
print('=== Insights Table — Final State ===')
print(f'Total rows: {len(df_insights_clean):,}')
print(f'\nInsight type breakdown:')
print(df_insights_clean['insight_type'].value_counts())
print(f'\nsource_score stats (after filtering):')
print(df_insights_clean['source_score'].describe().round(3))

print()
print('=== Cleaning Summary ===')
summary_data = {
    'Step': [
        'Remove no-URL rows',
        'Remove missing champion',
        'Exclude no-transcript rows',
        'Remove non-English titles',
        'Fix message_timestamp dtype',
        'Filter low source_score insights (<0.40)',
        'Dedup near-duplicate insights (cosine sim)'
    ],
    'Table': ['videos','videos','videos','videos','videos','insights','insights'],
    'Action': ['Drop rows','Drop rows','Exclude from analysis','Drop rows','Type conversion','Filter rows','Merge duplicates'],
}
print(pd.DataFrame(summary_data).to_string(index=False))